# Baseline reproduction on GPU (Colab or Kaggle)

Reproduces CausalPFN (NeurIPS 2025) normalized Qini on **Hillstrom** and **Criteo**. Linear: **TEA-6**.

**Before running**
- **Colab:** Runtime → Change runtime type → **T4 GPU** (or A100 on Colab Pro).
- **Kaggle:** Settings → Accelerator → **GPU T4 x2**, and turn **Internet on**.

Run the cells top to bottom. Change `RUN` in step 3 to choose what to reproduce. Re-running continues where it stopped: finished folds are skipped.

## 1. Environment and GPU check

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
IN_KAGGLE = os.path.exists("/kaggle")
print("Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Other machine")

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU found. Switch the runtime to a GPU first.")
print("GPU:", gpu.stdout.strip())

## 2. Where results are saved

Files in the notebook's working folder disappear when the session ends.
- **Colab:** saved to Google Drive (`MyDrive/cue-outputs`). You'll be asked to allow access.
- **Kaggle:** saved to `/kaggle/working/outputs`. Use **Save Version** at the end to keep them.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/cue-outputs")
    WORK = Path("/content")
elif IN_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working/outputs")
    WORK = Path("/kaggle/working")
else:
    OUTPUT_DIR = Path.cwd() / "outputs"
    WORK = Path.cwd()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUTPUT_DIR)

## 3. Choose what to run

| `RUN` | Paper | Time on a T4 (rough) |
|---|---|---|
| `"hillstrom"` | Table 2, both Hillstrom arms | under 1 hour, **run this first** |
| `"criteo"` | Table 2, Criteo 50k sample | about 1 hour |
| `"criteo_full"` | Table 7, full Criteo | many hours; likely longer than one free session |

In [ ]:
RUN = "hillstrom"   # "hillstrom" | "criteo" | "criteo_full"

COMMANDS = {
    "hillstrom": "bash scripts/run_baseline_hillstrom.sh",
    "criteo": "bash scripts/run_baseline_criteo.sh",
    "criteo_full": "bash scripts/run_baseline_criteo.sh --include-full",
}
assert RUN in COMMANDS, f"RUN must be one of {list(COMMANDS)}"
print("Will run:", COMMANDS[RUN])

## 4. Get the code (with the authors' pinned CausalPFN source)

In [ ]:
REPO = WORK / "causal-uplift-engine"
BRANCH = "baseline-reproduction"

def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, executable="/bin/bash")

if not REPO.exists():
    sh(f"git clone --recurse-submodules -b {BRANCH} https://github.com/yashashree5/causal-uplift-engine.git {REPO}")
else:
    sh(f"git fetch origin && git checkout {BRANCH} && git pull --ff-only && git submodule update --init --recursive", cwd=REPO)

sh("git log --oneline -1 && git -C third_party/CausalPFN log --oneline -1", cwd=REPO)

## 5. Install the pinned environment (Python 3.10)

The authors' setup uses Python 3.10. This creates a separate 3.10 environment so the notebook's own Python version doesn't matter. Takes a few minutes.

In [ ]:
VENV = REPO / ".venv-baseline"
if not (VENV / "bin" / "python").exists():
    sh("pip install -q uv")
    sh(f"uv venv -p 3.10 --seed {VENV}")
    sh(f"{VENV}/bin/pip install -q -r baseline/requirements.txt", cwd=REPO)
    sh(f"{VENV}/bin/pip install -q -e . --no-deps", cwd=REPO)

sh(f"{VENV}/bin/python -c \"import torch, econml, sklift; "
   f"print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| econml', econml.__version__)\"")

## 6. Run the baseline

Output streams below and is also saved under `outputs/logs/`. If the session stops, re-run cells 1–2 and 4–6: finished folds are skipped.

If you see **`CUDA out of memory`**, stop, note the GPU in TEA-6, and switch to a larger GPU (A100). Don't lower the context size, because the setup would no longer match the paper.

In [ ]:
env = dict(os.environ, OUTPUT_DIR=str(OUTPUT_DIR), PATH=f"{VENV}/bin:" + os.environ["PATH"],
           VIRTUAL_ENV=str(VENV), PYTHONUNBUFFERED="1")
proc = subprocess.Popen(COMMANDS[RUN], shell=True, cwd=REPO, env=env, executable="/bin/bash",
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nexit code:", proc.returncode)

## 7. Results

In [ ]:
dataset = "hillstrom" if RUN == "hillstrom" else "criteo"
report = REPO / "reports" / f"baseline_{dataset}.md"
if report.exists():
    from IPython.display import Markdown, display
    display(Markdown(report.read_text()))
    saved = OUTPUT_DIR / "reports"
    saved.mkdir(exist_ok=True)
    for f in (REPO / "reports").glob(f"baseline_{dataset}.*"):
        shutil.copy(f, saved / f.name)
    print("Copied report to", saved)
else:
    print("No report yet. Check the log above for errors.")

## 8. After the run

1. Download `reports/baseline_<dataset>.md` and `.csv` from the outputs folder.
2. Commit them to the `baseline-reproduction` branch with `Part of TEA-6` in the message.
3. Comment on **TEA-6** with: GPU name, total runtime, any row-count warning, and the CausalPFN numbers.